In [11]:
%%capture
# Install the official Groq library
!pip install groq

In [12]:
from groq import Groq
from google.colab import userdata

# Securely retrieve the Groq API key from Colab Secrets
try:
    groq_api_key = userdata.get('GROQ_API_KEY')
    client = Groq(api_key=groq_api_key)
    print("Groq client successfully initialized!")
except Exception as e:
    print(f"Error initializing Groq client: {e}\nPlease ensure GROQ_API_KEY is configured in your Secrets.")

Groq client successfully initialized!


In [13]:
# Initialize a global list to store the conversation history ONLY if it does not already exist
if 'chat_history' not in globals():
    chat_history = []

def llm(user_prompt):
    global chat_history
    system_prompt = """You are a helpful assistant."""

    # If the history is empty, initialize it with the system prompt
    if not chat_history and system_prompt:
        chat_history.append({"role": "system", "content": system_prompt})

    # Add the new user message to the history
    chat_history.append({"role": "user", "content": user_prompt})

    try:
        completion = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=chat_history,
            temperature=0.2
        )
        # Get the assistant's reply
        assistant_response = completion.choices[0].message.content

        # Add the assistant's response to the history so it's remembered next time
        chat_history.append({"role": "assistant", "content": assistant_response})

        return assistant_response
    except Exception as e:
        return f"An error occurred during API request: {e}"

### Food Nutrition MCP Server / Tool Registry
Below, we define a Mock MCP Server/Tool Registry that hosts tool definitions and executing functions for retrieving nutritional information of food items.

In [14]:
import json

# Mock Database representing our MCP server's knowledge base
NUTRITION_DATABASE = {
    "apple": {"calories": 52, "protein": "0.3g", "carbs": "14g", "fat": "0.2g", "fiber": "2.4g"},
    "banana": {"calories": 89, "protein": "1.1g", "carbs": "23g", "fat": "0.3g", "fiber": "2.6g"},
    "chicken breast": {"calories": 165, "protein": "31g", "carbs": "0g", "fat": "3.6g", "fiber": "0g"},
    "salmon": {"calories": 208, "protein": "20g", "carbs": "0g", "fat": "13g", "fiber": "0g"},
    "egg": {"calories": 78, "protein": "6g", "carbs": "0.6g", "fat": "5g", "fiber": "0g"}
}

def get_food_nutrition(food_name: str) -> str:
    """
    MCP Tool implementation: Fetches the nutritional details of a given food item.
    """
    food_lower = food_name.lower().strip()
    # Search for matches or substrings
    for key, info in NUTRITION_DATABASE.items():
        if key in food_lower or food_lower in key:
            return json.dumps({"food": key, "nutrition": info})
    return json.dumps({"error": f"Nutritional information for '{food_name}' was not found in our database."})

# Define the tool schema compatible with Groq / OpenAI tool calling
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_food_nutrition",
            "description": "Get the nutritional information (calories, protein, carbs, fat, fiber) for a specified food item.",
            "parameters": {
                "type": "object",
                "properties": {
                    "food_name": {
                        "type": "string",
                        "description": "The name of the food item (e.g., apple, banana, salmon)"
                    }
                },
                "required": ["food_name"]
            }
        }
    }
]

# Map function names to executable Python functions
available_tools = {
    "get_food_nutrition": get_food_nutrition
}

### Updated LLM Function with Tool-Calling (MCP integration)
We will now update the `llm` function to support tool execution. If the Groq LLM decides to call the nutrition tool, we will execute our local tool and send the response back to the LLM so it can formulate a final natural answer.

In [19]:
import json
from datetime import datetime

# Re-initialize or clear chat history if needed
chat_history = []

def log_event(event_type, details):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
    print(f"[{timestamp}] [{event_type}] {details}")

def llm_with_mcp(user_prompt):
    global chat_history
    system_prompt = """You are a helpful assistant. You have access to tools that fetch nutritional details.
Always use the provided tools if the user asks about the nutritional details of any food."""

    if not chat_history:
        chat_history.append({"role": "system", "content": system_prompt})

    chat_history.append({"role": "user", "content": user_prompt})

    log_event("USER_REQUEST", f"Prompt: {user_prompt}")

    try:
        log_event("LLM_REQUEST", f"Sending chat completion request with model: openai/gpt-oss-120b")
        # First call: Provide the prompt and the tool definitions
        response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=chat_history,
            tools=tools,
            tool_choice="auto",
            temperature=0.2
        )

        response_message = response.choices[0].message
        tool_calls = response_message.tool_calls

        log_event("LLM_RESPONSE", f"Received response. Tool calls detected: {True if tool_calls else False}")

        # If the model decided to call a tool (MCP simulation)
        if tool_calls:
            # Append model's reply with the tool request to history
            chat_history.append(response_message)

            # Handle each tool call requested by the model
            for tool_call in tool_calls:
                function_name = tool_call.function.name
                function_to_call = available_tools.get(function_name)
                function_args = json.loads(tool_call.function.arguments)

                log_event("MCP_SERVER_CALL", f"Executing '{function_name}' with arguments: {function_args}")
                tool_response = function_to_call(**function_args)
                log_event("MCP_SERVER_RESPONSE", f"Result: {tool_response}")

                # Append tool results back to history
                chat_history.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": tool_response
                })

            log_event("LLM_REQUEST_SECOND", "Sending execution results back to the LLM...")
            # Second call: Provide the tool execution results back to the LLM
            second_response = client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=chat_history
            )
            final_response = second_response.choices[0].message.content
            log_event("LLM_RESPONSE_SECOND", f"Final response generated: {final_response[:100]}...")
            chat_history.append({"role": "assistant", "content": final_response})
            return final_response

        else:
            # No tool call needed, return response directly
            final_response = response_message.content
            chat_history.append({"role": "assistant", "content": final_response})
            return final_response

    except Exception as e:
        log_event("ERROR", str(e))
        return f"An error occurred: {e}"

### Test the MCP integration
Let's test asking the LLM about an item in our database.

In [21]:
# Test fetching nutrition details
print(llm_with_mcp("What are the nutritional facts of a banana?"))

[2026-11-20 10:25:59.669] [USER_REQUEST] Prompt: What are the nutritional facts of a banana?
[2026-11-20 10:25:59.669] [LLM_REQUEST] Sending chat completion request with model: openai/gpt-oss-120b
[2026-11-20 10:26:00.626] [LLM_RESPONSE] Received response. Tool calls detected: False
Here’s a quick rundown of the typical nutrition you’ll find in a medium‑sized banana (about 118 g):

| Nutrient | Amount |
|----------|--------|
| **Calories** | 89 kcal |
| **Protein** | 1.1 g |
| **Carbohydrates** | 23 g |
| **• Sugars** | ~12 g |
| **• Dietary Fiber** | 2.6 g |
| **Fat** | 0.3 g |
| **• Saturated Fat** | 0.1 g |
| **Potassium** | ~358 mg |
| **Vitamin C** | 8.7 mg (≈10 % DV) |
| **Vitamin B6** | 0.4 mg (≈20 % DV) |
| **Magnesium** | 32 mg (≈8 % DV) |

*Values are averages; actual numbers can vary a bit depending on the banana’s size and ripeness.*
